In [ ]:
from itertools import cycle

import numpy as np
import pyemu
from pathlib import Path
import pypestvis as ppv

In [ ]:
m_d = Path("freyberg_ies")

## Read in the PEST control file and modify obs to get i,j,k cols

In [ ]:
pst = pyemu.Pst(str(m_d / "freyberg.pst"))
obs = pst.observation_data
obs.loc[obs.oname=='hds', ['k', 'i', 'j']] = obs.loc[obs.oname=='hds'].obgnme.str.rsplit("_",expand=True, n=3)[[1,2,3]].values
obs.loc[(obs.obgnme == 'hdar') & (obs.i.astype("Int32")>20), 'obgnme'] = 'hdar2'
obs.loc[(obs.obgnme == 'hdar'), 'weight'] = 0
othersel = ~obs.obgnme.str.startswith('hda')
obs.loc[othersel, 'obgnme'] = obs.loc[othersel, 'usecol']
pst.observation_data = obs

## Build a VisHandler from PEST control file and model directory
* reads in model spatial information
* loads ensembles if available
* organises output into mapable and unmapable and builds spatial ref
* builds plotly objects and ipywidget callbacks for visualisation

In [ ]:
vh = ppv.VisHandler(pst, wd=m_d, crs="epsg:32614")

In [ ]:
vh.set_mapsel_options()

In [ ]:
display(vh.default_map_layout)

In [ ]:
vh.map_ts

In [ ]:
vh.default_unmap_layout

In [ ]:
# vh.map_widget.data[1]

In [ ]:
import plotly.graph_objects as go
from itertools import cycle
from plotly.colors import DEFAULT_PLOTLY_COLORS
tsplot = go.Figure(layout=dict(margin_t=30,
                               margin_b=10,
                               margin_l=10,
                               margin_r=10,
                               width=600,
                               height=300,
                               title='T-series',
                               showlegend=False,
                               xaxis_autorange=True)
                   )
iters = sorted(vh.real_dict.keys())
ccycle = cycle(DEFAULT_PLOTLY_COLORS)
with tsplot.batch_update():
    for i, reals in vh.real_dict.items():
            if i == iters[0]:
                c = 'rgba(112,112,112,0.75)'
            elif i == iters[-1]:
                c = 'rgba(20,49,220,0.75)'
            else:
                c = next(ccycle)
            print(c)
            for r in reals:
                tsplot.add_trace(go.Scattergl(name=f"{i}_{r}",
                                              legendgroup=f'iter_{i}',mode='lines',
                                              line=dict(color=c, width=0.5),
                             hovertemplate=None, hoverinfo='none', opacity=0.5))

In [ ]:
obsovert = vh._tmp_map_kidxmap.loc[(vh._sel_cellid, slice(None))].sort_index()
x = obsovert.index
idx = obsovert.values
df = vh._tmp_map_gph.ens.loc[idx, :].set_index(x)

In [ ]:
%%timeit
# update method
with tsplot.batch_update():
    # vh.map_ts.data = []
    for dfi in df.T.itertuples():
        i, r = dfi[0] # iteration and real
        tsplot.update_traces(x=x, # defined above
                             y=dfi[1:], # itertuples so 0 is index
                             showlegend=True,
                             selector=dict(name=f'{i}_{r}'))

In [ ]:
tsplot

In [ ]:
%%timeit
# rewrite method
with tsplot.batch_update():
    tsplot.data = []
    for dfi in df.T.itertuples():
        i, r = dfi[0] # iteration and real
        if i == iters[0]:
            c = 'rgba(112,112,112,0.75)'
        elif i == iters[-1]:
            c = 'rgba(20,49,220,0.75)'
        else:
            c = next(ccycle)
        tsplot.add_trace(go.Scattergl(
            x=x, # defined above
            y=dfi[1:], # itertuples so 0 is index
            name=f"{i}_{r}", legendgroup=f'iter_{i}',mode='lines',
                         line=dict(color=c, width=0.5),
                         hovertemplate=None, hoverinfo='none', opacity=0.5,
                                     showlegend=True,
        ))
